In [14]:

from conch.open_clip_custom import create_model_from_pretrained, tokenize, get_tokenizer
import torch
import os
from PIL import Image
from pathlib import Path
import pandas as pd
from tqdm import tqdm
# remember to "conda activate conch"

In [6]:
model_cfg = 'conch_ViT-B-16'
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
checkpoint_path = 'checkpoints/conch/pytorch_model.bin'
model, preprocess = create_model_from_pretrained(model_cfg, checkpoint_path, device=device)
model = model.eval()

In [ ]:
classes = ['sessile serrated adenomas/polyps', 
           'hyperplastic polyps']

class_prompts = {
    "SSA": [
        "a histopathology image of a sessile serrated adenoma",
        "colonic polyp showing serrated architecture consistent with SSA",
        "H&E stained tissue section of sessile serrated lesion",
        "microscopic image of serrated colonic crypts with dilation",
    ],
    "HP": [
        "a histopathology image of a hyperplastic polyp",
        "colonic polyp with straight crypt architecture consistent with hyperplastic polyp",
        "H&E stained tissue showing hyperplastic polyp",
        "benign colonic mucosa with elongated crypts typical of hyperplastic polyp",
    ]
}

In [15]:
@torch.no_grad()
def encode_prompts(model, prompts_dict, tokenizer):
    class_embeddings = {}

    for cls, prompts in prompts_dict.items():

        # batch tokenize ALL prompts for this class
        tokenized_prompts = tokenize(
            texts=prompts,
            tokenizer=tokenizer
        ).to(device)

        # encode text
        text_features = model.encode_text(tokenized_prompts)

        # normalize
        text_features = text_features / text_features.norm(dim=-1, keepdim=True)

        # aggregate prompts → class embedding
        class_embedding = text_features.mean(dim=0)
        class_embedding = class_embedding / class_embedding.norm()

        class_embeddings[cls] = class_embedding

    return class_embeddings

@torch.no_grad()
def predict_image(image_path, model, text_embeddings):
    image = preprocess(Image.open(image_path).convert("RGB")).unsqueeze(0).to(device)

    image_features = model.encode_image(image)
    image_features = image_features / image_features.norm(dim=-1, keepdim=True)

    best_class = None
    best_score = -1

    for cls, text_feat in text_embeddings.items():
        score = (image_features @ text_feat.unsqueeze(0).T).item()

        if score > best_score:
            best_score = score
            best_class = cls

    return best_class, best_score

In [16]:
tokenizer = get_tokenizer()
text_embeddings = encode_prompts(model, class_prompts, tokenizer)

In [20]:
image_dir = "mhist_data/images"
annotations_dir = "mhist_data/annotations.csv"
df = pd.read_csv(annotations_dir)

preds = []
gts = []

for _, row in tqdm(df.iterrows(), total=len(df)):
    img_name = row["Image Name"]
    gt = row["Majority Vote Label"]

    img_path = os.path.join(image_dir, img_name)

    pred, score = predict_image(img_path, model, text_embeddings)

    preds.append(pred)
    gts.append(gt)

/home/linus-dannull/Documents/WashUCoding/Computer Vision/final_proj


100%|██████████| 3152/3152 [13:23<00:00,  3.92it/s]


In [21]:
from sklearn.metrics import accuracy_score, classification_report

print("Accuracy:", accuracy_score(gts, preds))
print(classification_report(gts, preds))

Accuracy: 0.5736040609137056
              precision    recall  f1-score   support

          HP       0.70      0.67      0.68      2162
         SSA       0.34      0.37      0.35       990

    accuracy                           0.57      3152
   macro avg       0.52      0.52      0.52      3152
weighted avg       0.58      0.57      0.58      3152



As can be seen in the accuracy of the model it behaves poorly when directly asked to classify an unseen dataset for labels it likely was not directly trained to detect. It's accuracy is only 57%. A model which always predicted HP would get around 68% accuracy. How can we adapt the output CONCH can give to yield better results.